In [2]:
"""
ATTENTION MECHANISM EXERCISE
============================

Scenario: We have a tiny sentence: "the cat sat down"
4 tokens, each represented as a 3-dimensional embedding vector.

In real models:
- GPT-2 uses 768-dim embeddings, 50,257 vocab size
- Here we use 3-dim embeddings, 4 tokens (so you can do the math)

YOUR TASK:
1. Compute Q, K, V matrices
2. Compute attention scores
3. Apply softmax
4. Compute the final output
"""

import numpy as np
np.set_printoptions(precision=4)

# ============================================================
# GIVEN: Token embeddings (4 tokens × 3 dimensions)
# ============================================================
# These come from nn.Embedding — each word maps to a learned vector.
# In real models this is learned. Here we just define them.

embeddings = np.array([
    [1.0, 0.5, 0.3],   # "the"
    [0.2, 0.8, 0.9],   # "cat"
    [0.6, 0.1, 0.7],   # "sat"
    [0.4, 0.3, 0.5],   # "down"
])
# Shape: (4, 3) — 4 tokens, each is a 3-dim vector

print("Embeddings (4 tokens × 3 dims):")
print(embeddings)
print()

Embeddings (4 tokens × 3 dims):
[[1.  0.5 0.3]
 [0.2 0.8 0.9]
 [0.6 0.1 0.7]
 [0.4 0.3 0.5]]



In [3]:
# ============================================================
# GIVEN: Weight matrices W_q, W_k, W_v (3×3 each)
# ============================================================
# These are the nn.Linear layers. In real models, these are LEARNED
# through backpropagation. Here we define them manually.
#
# How are these created in PyTorch?
#   self.W_q = nn.Linear(3, 3, bias=False)  # that's it
#   self.W_k = nn.Linear(3, 3, bias=False)
#   self.W_v = nn.Linear(3, 3, bias=False)
#
# Each is a (3×3) matrix because input_dim=3, output_dim=3
# In GPT-2 these would be (768×768)

W_q = np.array([
    [0.1, 0.2, 0.0],
    [0.3, 0.1, 0.4],
    [0.5, 0.0, 0.2],
])

W_k = np.array([
    [0.2, 0.0, 0.3],
    [0.1, 0.3, 0.1],
    [0.0, 0.2, 0.4],
])

W_v = np.array([
    [0.4, 0.1, 0.0],
    [0.0, 0.3, 0.2],
    [0.1, 0.0, 0.5],
])

print("W_q (query weights):"); print(W_q); print()
print("W_k (key weights):"); print(W_k); print()
print("W_v (value weights):"); print(W_v); print()

W_q (query weights):
[[0.1 0.2 0. ]
 [0.3 0.1 0.4]
 [0.5 0.  0.2]]

W_k (key weights):
[[0.2 0.  0.3]
 [0.1 0.3 0.1]
 [0.  0.2 0.4]]

W_v (value weights):
[[0.4 0.1 0. ]
 [0.  0.3 0.2]
 [0.1 0.  0.5]]



In [4]:
# ============================================================
# TASK 1: Compute Q, K, V matrices
# ============================================================
# Formula:
#   Q = embeddings @ W_q    (each token gets a query vector)
#   K = embeddings @ W_k    (each token gets a key vector)
#   V = embeddings @ W_v    (each token gets a value vector)
#
# Shape: (4, 3) @ (3, 3) = (4, 3)
# Each row = one token's Q/K/V vector

Q = embeddings @ W_q
K = embeddings @ W_k
V = embeddings @ W_v

print("Q (queries for each token):"); print(Q); print()
print("K (keys for each token):"); print(K); print()
print("V (values for each token):"); print(V); print()

Q (queries for each token):
[[0.4  0.25 0.26]
 [0.71 0.12 0.5 ]
 [0.44 0.13 0.18]
 [0.38 0.11 0.22]]

K (keys for each token):
[[0.25 0.21 0.47]
 [0.12 0.42 0.5 ]
 [0.13 0.17 0.47]
 [0.11 0.19 0.35]]

V (values for each token):
[[0.43 0.25 0.25]
 [0.17 0.26 0.61]
 [0.31 0.09 0.37]
 [0.21 0.13 0.31]]



In [5]:
# ============================================================
# TASK 2: Compute attention scores
# ============================================================
# Formula: scores = Q @ K^T
#
# Shape: (4, 3) @ (3, 4) = (4, 4)
#
# scores[i][j] = "how much should token i attend to token j?"
# For example, scores[1][0] = how much "cat" attends to "the"
#
# Hint: K^T means K transposed. In numpy: K.T

scores = Q @ K.T

print("Raw attention scores (4×4):")
print("Rows = query token, Columns = key token")
print("       the    cat    sat    down")
print(scores); print()

Raw attention scores (4×4):
Rows = query token, Columns = key token
       the    cat    sat    down
[[0.2747 0.283  0.2167 0.1825]
 [0.4377 0.3856 0.3477 0.2759]
 [0.2219 0.1974 0.1639 0.1361]
 [0.2215 0.2018 0.1715 0.1397]]



In [6]:
import math

# ============================================================
# TASK 3: Scale the scores
# ============================================================
# Formula: scaled_scores = scores / sqrt(d_k)
#
# d_k = dimension of key vectors = 3
# Why? Without scaling, large dimensions cause large dot products,
# pushing softmax into regions with tiny gradients (vanishing gradient).
#
# sqrt(3) ≈ 1.732

d_k = 3
scaled_scores = scores / math.sqrt(d_k)

print("Scaled attention scores:")
print(scaled_scores); print()

Scaled attention scores:
[[0.1586 0.1634 0.1251 0.1054]
 [0.2527 0.2226 0.2007 0.1593]
 [0.1281 0.114  0.0946 0.0786]
 [0.1279 0.1165 0.099  0.0807]]



In [7]:
# ============================================================
# TASK 4: Apply softmax (row-wise)
# ============================================================
# Each ROW gets its own softmax — because each token has its own
# probability distribution over all other tokens.
#
# softmax(x_i) = exp(x_i) / sum(exp(x_j)) for all j in the row
#
# After softmax, each row sums to 1.0

def softmax(x):
    """Apply softmax row-wise"""
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))  # subtract max for numerical stability
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

attention_weights = softmax(scaled_scores)

print("Attention weights (after softmax):")
print("Each row sums to 1.0 — it's a probability distribution")
print("       the    cat    sat    down")
print(attention_weights)
print("Row sums:", attention_weights.sum(axis=1))  # should all be 1.0
print()

Attention weights (after softmax):
Each row sums to 1.0 — it's a probability distribution
       the    cat    sat    down
[[0.2551 0.2563 0.2467 0.2419]
 [0.2611 0.2533 0.2478 0.2378]
 [0.2561 0.2525 0.2477 0.2437]
 [0.2555 0.2526 0.2482 0.2437]]
Row sums: [1. 1. 1. 1.]



In [8]:
# ============================================================
# TASK 5: Compute the output
# ============================================================
# Formula: output = attention_weights @ V
#
# Shape: (4, 4) @ (4, 3) = (4, 3)
#
# Each output row is a WEIGHTED MIX of all value vectors.
# Token "cat" output = 0.22*V("the") + 0.28*V("cat") + 0.25*V("sat") + 0.25*V("down")
# (example numbers, yours will differ)
#
# This is the "context-aware" representation — each token now
# contains information from tokens it attended to.

output = attention_weights @ V

print("Output (context-aware representations):")
print(output); print()

Output (context-aware representations):
[[0.2805 0.1841 0.3864]
 [0.2821 0.1843 0.3852]
 [0.281  0.1837 0.3852]
 [0.2809 0.1836 0.3853]]



In [9]:
# ============================================================
# TASK 6: Interpret the results
# ============================================================
# Look at the attention_weights matrix.
# Answer these questions (just think about them):
#
# 1. Which token does "cat" (row 1) attend to the most?
# 2. Which token does "sat" (row 2) attend to the most?
# 3. Are the output vectors different from the original embeddings?
#    Why? (Because each token now carries info from other tokens)
# 4. What would happen if attention weights were all 0.25 (uniform)?
#    (Answer: output = simple average of all V vectors, no "attention" at all)


# ============================================================
# BONUS: Compare with uniform attention (no attention at all)
# ============================================================
uniform_weights = np.ones((4, 4)) / 4  # every token equally important
uniform_output = uniform_weights @ V
print("Output with UNIFORM attention (no real attention):")
print(uniform_output)
print("Notice: every row is identical — no token is special")
print()
print("Output with LEARNED attention:")
print(output)
print("Notice: every row is different — each token has its own context")

Output with UNIFORM attention (no real attention):
[[0.28   0.1825 0.385 ]
 [0.28   0.1825 0.385 ]
 [0.28   0.1825 0.385 ]
 [0.28   0.1825 0.385 ]]
Notice: every row is identical — no token is special

Output with LEARNED attention:
[[0.2805 0.1841 0.3864]
 [0.2821 0.1843 0.3852]
 [0.281  0.1837 0.3852]
 [0.2809 0.1836 0.3853]]
Notice: every row is different — each token has its own context


In [10]:
# Vocab projection layer (3-dim → 4 words)
W_vocab = np.array([[1.2, 0.1, -0.3, 0.5],
                     [-0.4, 0.9, 0.7, -0.2],
                     [0.2, 0.3, 1.1, 0.8]])

# Take last token's output, project to vocab size
last_token = output[-1]              # shape: (3,)
logits = last_token @ W_vocab        # shape: (4,)
probs = softmax(logits)              # shape: (4,)

vocab = ["the", "cat", "sat", "down"]
for word, prob in zip(vocab, probs):
    print(f"{word:>6}: {prob:.4f}")

print(f"\nPredicted next word: {vocab[np.argmax(probs)]}")

   the: 0.2393
   cat: 0.2318
   sat: 0.2718
  down: 0.2570

Predicted next word: sat
